In [ ]:
"""
AS/400 Phase 1 – Transform (Parquet-basiert)
=============================================
Liest Parquet-Files (aus as400_phase1_raw_extract.py)
und erzeugt eine Analyse-Excel für as400_phase1_analyse.ipynb.

Erwartet folgende Parquet-Files im gleichen Verzeichnis:
    systables.parquet
    syscolumns.parquet
    systablestat.parquet
    syscolumnstat.parquet
    sysviews.parquet
    sysindexes.parquet
    sysroutines.parquet
    syskeys.parquet          (optional)

Fehlende Files werden übersprungen – kein Abbruch.

Voraussetzungen:
    pip install pandas openpyxl pyarrow

Nutzung:
    python as400_phase1_transform_parquet.py
    (optional: Verzeichnis als Argument)
    python as400_phase1_transform_parquet.py C:/pfad/zu/parquet/files
"""

import re
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime
from pathlib import Path
import sys


# ─────────────────────────────────────────────
# KONFIGURATION
# ─────────────────────────────────────────────

# Verzeichnis der Parquet-Files (Argument oder aktuelles Verzeichnis)
PARQUET_DIR = Path(sys.argv[1]) if len(sys.argv) > 1 else Path(".")

OUTPUT_FILE = f"as400_analyse_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

# Mapping: interner Name → Dateiname (ohne .parquet)
PARQUET_FILES = {
    "systables":    "systables",
    "syscolumns":   "syscolumns",
    "systablestat": "systablestat",
    "syscolumnstat":"syscolumnstat",
    "sysviews":     "sysviews",
    "sysindexes":   "sysindexes",
    "sysroutines":  "sysroutines",
    "syskeys":      "syskeys",       # optional
}


# ─────────────────────────────────────────────
# LADEN
# ─────────────────────────────────────────────

# Binärspalten die wir nicht brauchen und die Encoding-Fehler verursachen
DROP_COLS = {
    "syscolumnstat": ["INTERNAL_STATISTICS_ID"],
    "sysroutines":   ["PARM_SIGNATURE", "PARSE_TREE", "PARM_ARRAY"],
}

# Steuerzeichen entfernen (0x00–0x1F außer Tab \x09 und Newline \x0A)
_CTRL_CHARS = re.compile(r"[\x00-\x08\x0b-\x1f\x7f]")

def _clean_value(v):
    """Bytes → utf-8 str, dann Steuerzeichen via Regex entfernen."""
    if isinstance(v, bytes):
        v = v.decode("utf-8", errors="replace")
    if isinstance(v, str):
        v = _CTRL_CHARS.sub("", v).strip()
    return v

def load_parquets():
    raw = {}
    print(f"\nLade Parquet-Files aus: {PARQUET_DIR.resolve()}\n")
    for key, filename in PARQUET_FILES.items():
        path = PARQUET_DIR / f"{filename}.parquet"
        if not path.exists():
            print(f"  {filename}.parquet   → nicht gefunden, wird übersprungen")
            raw[key] = pd.DataFrame()
            continue
        try:
            df = pd.read_parquet(path)

            # Binärspalten droppen (vor Bereinigung, sonst Encoding-Fehler)
            drop = [c for c in DROP_COLS.get(key, []) if c in df.columns]
            if drop:
                df = df.drop(columns=drop)
                print(f"  {filename}.parquet   → Binärspalten entfernt: {drop}")

            # String-Spalten bereinigen: Bytes → utf-8, Steuerzeichen raus
            for col in df.select_dtypes(include="object").columns:
                df[col] = df[col].apply(_clean_value)

            print(f"  {filename}.parquet   → {len(df):>7,} Zeilen | {len(df.columns)} Spalten")
            raw[key] = df
        except Exception as e:
            print(f"  {filename}.parquet   → FEHLER: {e}")
            raw[key] = pd.DataFrame()
    return raw


# ─────────────────────────────────────────────
# HILFSFUNKTION: sicheres Umbenennen
# (nur Spalten umbenennen die wirklich vorhanden sind)
# ─────────────────────────────────────────────

def safe_rename(df, rename_map, keep=None):
    """Benennt nur vorhandene Spalten um; keep=None → alle behalten."""
    cols_present = {k: v for k, v in rename_map.items() if k in df.columns}
    result = df.rename(columns=cols_present)
    if keep:
        keep_present = [v for v in keep if v in result.columns]
        result = result[keep_present]
    return result


def to_numeric_safe(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


# ─────────────────────────────────────────────
# TRANSFORM-FUNKTIONEN
# ─────────────────────────────────────────────

def t_schema_uebersicht(raw):
    df = raw.get("systables", pd.DataFrame())
    if df.empty or "TABLE_SCHEMA" not in df.columns:
        return pd.DataFrame()
    agg = (
        df.assign(
            ist_tabelle  = df["TABLE_TYPE"].eq("T").astype(int),
            ist_view     = df["TABLE_TYPE"].eq("V").astype(int),
            ist_alias    = df["TABLE_TYPE"].eq("A").astype(int),
            ist_physisch = df["TABLE_TYPE"].eq("P").astype(int),
        )
        .groupby("TABLE_SCHEMA", as_index=False)
        .agg(
            anzahl_gesamt = ("TABLE_NAME",   "count"),
            tabellen      = ("ist_tabelle",  "sum"),
            views         = ("ist_view",     "sum"),
            aliases       = ("ist_alias",    "sum"),
            physisch      = ("ist_physisch", "sum"),
        )
        .rename(columns={"TABLE_SCHEMA": "schema_name"})
        .sort_values("anzahl_gesamt", ascending=False)
    )
    return agg


def t_tabellen(raw):
    tabs = raw.get("systables", pd.DataFrame())
    stat = raw.get("systablestat", pd.DataFrame())
    if tabs.empty:
        return pd.DataFrame()

    t = tabs[tabs["TABLE_TYPE"].isin(["T", "P"])].copy() if "TABLE_TYPE" in tabs.columns else tabs.copy()
    t = to_numeric_safe(t, ["COLUMN_COUNT"])

    rename = {
        "TABLE_SCHEMA":           "schema_name",
        "TABLE_NAME":             "tabelle",
        "TABLE_TYPE":             "typ",
        "TABLE_TEXT":             "beschreibung",
        "COLUMN_COUNT":           "spalten_anzahl",
        "LAST_ALTERED_TIMESTAMP": "letzte_aenderung",
        "SYSTEM_TABLE_NAME":      "systemname",
    }
    t = safe_rename(t, rename, keep=list(rename.values()))

    if not stat.empty:
        s = to_numeric_safe(stat.copy(), ["NUMBER_ROWS", "DATA_SIZE"])
        s = safe_rename(s, {
            "TABLE_SCHEMA": "schema_name",
            "TABLE_NAME":   "tabelle",
            "NUMBER_ROWS":  "zeilen_anzahl",
            "DATA_SIZE":    "daten_bytes",
        }, keep=["schema_name","tabelle","zeilen_anzahl","daten_bytes"])
        s["daten_kb"] = s["daten_bytes"].fillna(0) / 1024
        s["daten_mb"] = s["daten_bytes"].fillna(0) / 1024 / 1024
        s = s.drop(columns=["daten_bytes"], errors="ignore")
        t = t.merge(s, on=["schema_name","tabelle"], how="left")
        t["zeilen_anzahl"] = t["zeilen_anzahl"].fillna(0).astype(int)
        t["daten_kb"]      = t["daten_kb"].fillna(0).round(2)
        t["daten_mb"]      = t["daten_mb"].fillna(0).round(4)

    return t.sort_values(["schema_name","tabelle"]).reset_index(drop=True)


def t_spalten(raw):
    df = raw.get("syscolumns", pd.DataFrame())
    if df.empty:
        return pd.DataFrame()
    df = to_numeric_safe(df.copy(), ["ORDINAL_POSITION","LENGTH","NUMERIC_SCALE"])
    rename = {
        "TABLE_SCHEMA":       "schema_name",
        "TABLE_NAME":         "tabelle",
        "ORDINAL_POSITION":   "position",
        "COLUMN_NAME":        "spalte",
        "SYSTEM_COLUMN_NAME": "system_spalte",
        "DATA_TYPE":          "datentyp",
        "LENGTH":             "laenge",
        "NUMERIC_SCALE":      "dezimalstellen",
        "IS_NULLABLE":        "nullable",
        "COLUMN_DEFAULT":     "standard_wert",
        "COLUMN_TEXT":        "beschreibung",
        "HAS_DEFAULT":        "hat_default",
    }
    return safe_rename(df, rename, keep=list(rename.values())).sort_values(
        ["schema_name","tabelle","position"]
    ).reset_index(drop=True)


def t_views(raw):
    views = raw.get("sysviews", pd.DataFrame())
    tabs  = raw.get("systables", pd.DataFrame())
    if views.empty:
        return pd.DataFrame()
    v = safe_rename(views, {
        "TABLE_SCHEMA":    "schema_name",
        "TABLE_NAME":      "view_name",
        "VIEW_DEFINITION": "view_definition",
    })
    if not tabs.empty:
        t = safe_rename(tabs, {
            "TABLE_SCHEMA": "schema_name",
            "TABLE_NAME":   "view_name",
            "TABLE_TEXT":   "beschreibung",
            "COLUMN_COUNT": "spalten_anzahl",
        }, keep=["schema_name","view_name","beschreibung","spalten_anzahl"])
        v = v.merge(t, on=["schema_name","view_name"], how="left")
    return v.sort_values(["schema_name","view_name"]).reset_index(drop=True)


def t_indizes(raw):
    df = raw.get("sysindexes", pd.DataFrame())
    if df.empty:
        return pd.DataFrame()
    df = to_numeric_safe(df.copy(), ["NUMBER_OF_COLUMNS"])
    return safe_rename(df, {
        "TABLE_SCHEMA":      "schema_name",
        "TABLE_NAME":        "tabelle",
        "INDEX_NAME":        "index_name",
        "IS_UNIQUE":         "eindeutig",
        "INDEX_TYPE":        "index_typ",
        "NUMBER_OF_COLUMNS": "anzahl_spalten",
        "SYSTEM_INDEX_NAME": "system_name",
        "INDEX_TEXT":        "beschreibung",
    }, keep=["schema_name","tabelle","index_name","eindeutig",
             "index_typ","anzahl_spalten","system_name","beschreibung"]
    ).sort_values(["schema_name","tabelle","index_name"]).reset_index(drop=True)


def t_schluessel(raw):
    df = raw.get("syskeys", pd.DataFrame())
    if df.empty:
        return pd.DataFrame()
    df = to_numeric_safe(df.copy(), ["ORDINAL_POSITION"])
    return safe_rename(df, {
        "TABLE_SCHEMA":    "schema_name",
        "TABLE_NAME":      "tabelle",
        "INDEX_NAME":      "index_name",
        "COLUMN_NAME":     "spalte",
        "ORDINAL_POSITION":"position",
        "ORDERING":        "sortierung",
    }, keep=["schema_name","tabelle","index_name","spalte","position","sortierung"]
    ).sort_values(["schema_name","tabelle","index_name","position"]).reset_index(drop=True)


def t_tabellenstatistik(raw):
    df = raw.get("systablestat", pd.DataFrame())
    if df.empty:
        return pd.DataFrame()
    df = to_numeric_safe(df.copy(), ["NUMBER_ROWS","DATA_SIZE","OVERFLOW"])
    r = safe_rename(df, {
        "TABLE_SCHEMA":         "schema_name",
        "TABLE_NAME":           "tabelle",
        "NUMBER_ROWS":          "zeilen_anzahl",
        "DATA_SIZE":            "daten_bytes",
        "OVERFLOW":             "overflow_zeilen",
        "LAST_USED_TIMESTAMP":  "zuletzt_genutzt",
        "STATISTICS_TIMESTAMP": "statistik_stand",
    })
    r["daten_mb"]      = r["daten_bytes"].fillna(0) / 1024 / 1024
    r["daten_mb"]      = r["daten_mb"].round(3)
    r["zeilen_anzahl"] = r["zeilen_anzahl"].fillna(0).astype(int)
    return r.sort_values("zeilen_anzahl", ascending=False).reset_index(drop=True)


def t_spaltenstatistik(raw):
    df = raw.get("syscolumnstat", pd.DataFrame())
    if df.empty:
        return pd.DataFrame()
    df = to_numeric_safe(df.copy(), ["DISTINCT_COUNT","NULL_COUNT"])
    return safe_rename(df, {
        "TABLE_SCHEMA":         "schema_name",
        "TABLE_NAME":           "tabelle",
        "COLUMN_NAME":          "spalte",
        "DISTINCT_COUNT":       "eindeutige_werte",
        "NULL_COUNT":           "null_anzahl",
        "STATISTICS_TIMESTAMP": "statistik_stand",
    }, keep=["schema_name","tabelle","spalte",
             "eindeutige_werte","null_anzahl","statistik_stand"]
    ).sort_values(["schema_name","tabelle","spalte"]).reset_index(drop=True)


def t_programme(raw):
    df = raw.get("sysroutines", pd.DataFrame())
    if df.empty:
        return pd.DataFrame()
    return safe_rename(df, {
        "ROUTINE_SCHEMA":         "schema_name",
        "ROUTINE_NAME":           "programm",
        "ROUTINE_TYPE":           "typ",
        "EXTERNAL_NAME":          "externer_name",
        "LANGUAGE":               "sprache",
        "ROUTINE_TEXT":           "beschreibung",
        "LAST_ALTERED_TIMESTAMP": "letzte_aenderung",
        "CREATED":                "erstellt",
    }, keep=["schema_name","programm","typ","externer_name",
             "sprache","beschreibung","letzte_aenderung","erstellt"]
    ).sort_values(["schema_name","programm"]).reset_index(drop=True)


def t_wichtigkeit_score(tabs_df, stat_df, idx_df):
    if tabs_df.empty:
        return pd.DataFrame()
    t = tabs_df[["schema_name","tabelle"]].copy()
    t["spalten_anzahl"] = pd.to_numeric(
        tabs_df.get("spalten_anzahl", 0), errors="coerce"
    ).fillna(0)

    if not stat_df.empty and "zeilen_anzahl" in stat_df.columns:
        s = stat_df[["schema_name","tabelle","zeilen_anzahl"]].copy()
        if "daten_mb" in stat_df.columns:
            s["daten_mb"] = stat_df["daten_mb"]
        t = t.merge(s, on=["schema_name","tabelle"], how="left")
    else:
        t["zeilen_anzahl"] = 0

    if not idx_df.empty and "tabelle" in idx_df.columns:
        i = idx_df.groupby(["schema_name","tabelle"]).size().reset_index(name="index_anzahl")
        t = t.merge(i, on=["schema_name","tabelle"], how="left")
    else:
        t["index_anzahl"] = 0

    t["zeilen_anzahl"] = t["zeilen_anzahl"].fillna(0)
    t["index_anzahl"]  = t["index_anzahl"].fillna(0)

    for col in ["zeilen_anzahl","spalten_anzahl","index_anzahl"]:
        mx = t[col].max()
        t[f"{col}_norm"] = (t[col] / mx).round(4) if mx > 0 else 0.0

    t["wichtigkeit_score"] = (
        t["zeilen_anzahl_norm"]  * 0.5 +
        t["index_anzahl_norm"]   * 0.3 +
        t["spalten_anzahl_norm"] * 0.2
    ).round(4)

    return t.sort_values("wichtigkeit_score", ascending=False).reset_index(drop=True)


# ─────────────────────────────────────────────
# EXCEL SCHREIBEN – in einem einzigen Durchgang
# (kein load_workbook nach to_excel)
# ─────────────────────────────────────────────

TAB_COLORS = ["2E75B6","00B050","7030A0","FF8C00","C00000",
              "4BACC6","F79646","0070C0","70AD47","1F4E79"]

SHEET_DESCRIPTIONS = {
    "00_Schema_Übersicht":  "Aggregation pro Schema/Library",
    "01_Tabellen":          "Tabellen + Statistik (Zeilen, MB)",
    "02_Spalten":           "Alle Spalten aller Tabellen",
    "03_Views":             "Views inkl. Definition",
    "04_Indizes":           "Alle Indizes",
    "05_Schlüssel":         "Index-Schlüsselspalten (wenn vorhanden)",
    "06_Tabellenstatistik": "Zeilen & Größe (sortiert)",
    "07_Spaltenstatistik":  "Distinct-Werte & NULL-Anteile",
    "08_Programme":         "Programme / Routinen",
    "09_Wichtigkeit_Score": "Gewichteter Score für Heatmap",
}

SHEET_SOURCES = {
    "00_Schema_Übersicht":  "systables",
    "01_Tabellen":          "systables + systablestat",
    "02_Spalten":           "syscolumns",
    "03_Views":             "sysviews + systables",
    "04_Indizes":           "sysindexes",
    "05_Schlüssel":         "syskeys",
    "06_Tabellenstatistik": "systablestat",
    "07_Spaltenstatistik":  "syscolumnstat",
    "08_Programme":         "sysroutines",
    "09_Wichtigkeit_Score": "systables + systablestat + sysindexes",
}

# Gemeinsame Style-Objekte (einmal erstellen, überall wiederverwenden)
H_FILL   = PatternFill("solid", fgColor="1F4E79")
ALT_FILL = PatternFill("solid", fgColor="EBF3FB")
H_FONT   = Font(name="Arial", bold=True, color="FFFFFF", size=10)
D_FONT   = Font(name="Arial", size=10)
THIN     = Side(style="thin", color="D0D0D0")
BORDER   = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
C_ALIGN  = Alignment(horizontal="center", vertical="center")
L_ALIGN  = Alignment(horizontal="left",   vertical="center")
R_ALIGN  = Alignment(horizontal="right",  vertical="center")


def _col_width(values, header):
    """Spaltenbreite aus Header + max. 200 Datenwerten schätzen."""
    sample = [str(v) for v in values[:200] if v is not None]
    max_data = max((len(s) for s in sample), default=0)
    return min(max(len(str(header)), max_data) + 4, 52)


def write_data_sheet(wb, sheet_name, df, tab_color):
    """Schreibt ein DataFrame direkt mit openpyxl – kein to_excel."""
    ws = wb.create_sheet(title=sheet_name[:31])
    ws.sheet_properties.tabColor = tab_color
    ws.freeze_panes = "A2"
    ws.row_dimensions[1].height = 20

    if df.empty:
        ws.cell(1, 1, "(keine Daten – Quelle nicht verfügbar)").font = D_FONT
        return

    # Header
    for col_idx, col_name in enumerate(df.columns, 1):
        cell = ws.cell(1, col_idx, str(col_name))
        cell.font      = H_FONT
        cell.fill      = H_FILL
        cell.border    = BORDER
        cell.alignment = C_ALIGN

    # Daten – Werte sicher nach Python-Typen konvertieren
    for row_idx, row in enumerate(df.itertuples(index=False), 2):
        fill = ALT_FILL if row_idx % 2 == 0 else None
        for col_idx, value in enumerate(row, 1):
            # numpy-Typen → Python-Typen (openpyxl braucht native Typen)
            if hasattr(value, "item"):
                value = value.item()
            # NaN/NaT → None
            if value != value:   # NaN-Check
                value = None
            cell = ws.cell(row_idx, col_idx, value)
            cell.font      = D_FONT
            cell.border    = BORDER
            cell.alignment = L_ALIGN
            if fill:
                cell.fill = fill

    # Spaltenbreiten
    for col_idx, col_name in enumerate(df.columns, 1):
        col_values = df.iloc[:, col_idx - 1].tolist()
        ws.column_dimensions[get_column_letter(col_idx)].width = _col_width(col_values, col_name)


def write_info_sheet(wb, transforms):
    ws = wb.create_sheet("00_Info")
    # Info-Sheet an erste Position
    wb.move_sheet(ws, offset=-(len(wb.sheetnames) - 1))
    ws.sheet_properties.tabColor = "1F4E79"

    ws["A1"] = "AS/400 Phase 1 – Aufbereitete Analyse-Daten"
    ws["A1"].font = Font(name="Arial", bold=True, size=14, color="1F4E79")

    for r, (label, value) in enumerate([
        ("Erstellt am:",         datetime.now().strftime("%d.%m.%Y %H:%M")),
        ("Parquet-Verzeichnis:", str(PARQUET_DIR.resolve())),
    ], 3):
        ws.cell(r, 1, label).font = Font(name="Arial", bold=True, size=10)
        ws.cell(r, 2, value).font = D_FONT

    # Tabellen-Header
    for col_idx, label in enumerate(["SHEET","BESCHREIBUNG","DATENSÄTZE","PARQUET-QUELLE"], 1):
        c = ws.cell(6, col_idx, label)
        c.font = H_FONT
        c.fill = H_FILL
        c.alignment = C_ALIGN
        c.border = BORDER

    for row_idx, (sheet, desc) in enumerate(SHEET_DESCRIPTIONS.items(), 7):
        df  = transforms.get(sheet, pd.DataFrame())
        src = SHEET_SOURCES.get(sheet, "")
        for col_idx, (val, align) in enumerate([
            (sheet,    L_ALIGN),
            (desc,     L_ALIGN),
            (len(df),  R_ALIGN),
            (src,      L_ALIGN),
        ], 1):
            c = ws.cell(row_idx, col_idx, val)
            c.font      = D_FONT
            c.alignment = align
            c.border    = BORDER

    ws.column_dimensions["A"].width = 26
    ws.column_dimensions["B"].width = 40
    ws.column_dimensions["C"].width = 14
    ws.column_dimensions["D"].width = 38

    note = len(SHEET_DESCRIPTIONS) + 9
    ws.cell(note, 1, "Nächster Schritt:").font = Font(name="Arial", bold=True, size=10)
    ws.cell(note+1, 1, "→ as400_phase1_analyse.ipynb öffnen").font = Font(
        name="Arial", size=10, color="2E75B6"
    )


def main():
    raw = load_parquets()

    print("\nTransformiere ...")
    tabs_df = t_tabellen(raw)
    stat_df = t_tabellenstatistik(raw)
    idx_df  = t_indizes(raw)

    transforms = {
        "00_Schema_Übersicht":  t_schema_uebersicht(raw),
        "01_Tabellen":          tabs_df,
        "02_Spalten":           t_spalten(raw),
        "03_Views":             t_views(raw),
        "04_Indizes":           idx_df,
        "05_Schlüssel":         t_schluessel(raw),
        "06_Tabellenstatistik": stat_df,
        "07_Spaltenstatistik":  t_spaltenstatistik(raw),
        "08_Programme":         t_programme(raw),
        "09_Wichtigkeit_Score": t_wichtigkeit_score(tabs_df, stat_df, idx_df),
    }

    print()
    for name, df in transforms.items():
        status = f"{len(df):>7,} Zeilen" if not df.empty else "—  leer / Quelle fehlt"
        print(f"  {name:<28} {status}")

    print(f"\nSchreibe: {OUTPUT_FILE} ...")
    wb = Workbook()
    wb.remove(wb.active)   # leeres Standard-Sheet entfernen

    for i, (name, df) in enumerate(transforms.items()):
        write_data_sheet(wb, name, df, TAB_COLORS[i % len(TAB_COLORS)])

    write_info_sheet(wb, transforms)
    wb.save(OUTPUT_FILE)

    print(f"✓ Fertig: {OUTPUT_FILE}")
    print(f"\nNächster Schritt: as400_phase1_analyse.ipynb öffnen")
    print(f"  Das Notebook sucht automatisch nach as400_analyse_*.xlsx\n")


if __name__ == "__main__":
    main()